In [ ]:
import pandas as pd
file_path = r"C:\Users\9191h\OneDrive\0. Work\2. 국어 강사\0. 학생 관리\[중1] - 1학기 진도 사항\weekly_message\0626_금.xlsx"

df = pd.read_excel(file_path)

df.head()

,날짜,이름,주테\n점수,독서1~3,독서4~6,비문학,문학,문법,19번,20번,과제,수업,전달 사항,최종문자
0,2026-06-21,다은이,87,1.0,NaN,1.0,10,NaN,NaN,NaN,NaN,NaN,다음 주부터 2026-2학기가 시작됩니다. 승급 및 반 이동이 진행되는 만큼 등원 ...,▷ 다은이는 이번 주간 테스트에서 87점을 받았습니다.\n\n- 독서 : 4/5\n...
1,2026-06-21,민지,83,1.0,NaN,NaN,"10, 11",18.0,NaN,NaN,NaN,NaN,다음 주부터 2026-2학기가 시작됩니다. 승급 및 반 이동이 진행되는 만큼 등원 ...,▷ 민지는 이번 주간 테스트에서 83점을 받았습니다.\n\n- 독서 : 4/5\n-...
2,2026-06-21,서윤이,96,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,다음 주부터 2026-2학기가 시작됩니다. 승급 및 반 이동이 진행되는 만큼 등원 ...,▷ 서윤이는 이번 주간 테스트에서 96점을 받았습니다.\n\n- 독서 : 5/5\n...
3,2026-06-21,차형이,75,1.0,1.0,1.0,"10, 11, 13",NaN,NaN,NaN,NaN,NaN,다음 주부터 2026-2학기가 시작됩니다. 승급 및 반 이동이 진행되는 만큼 등원 ...,▷ 차형이는 이번 주간 테스트에서 75점을 받았습니다.\n\n- 독서 : 3/5\n...
4,2026-06-21,다현이,87,1.0,NaN,1.0,NaN,NaN,NaN,2.0,NaN,NaN,다음 주부터 2026-2학기가 시작됩니다. 승급 및 반 이동이 진행되는 만큼 등원 ...,▷ 다현이는 이번 주간 테스트에서 87점을 받았습니다.\n\n- 독서 : 4/5\n...


In [2]:
question_desc = {
    10: "시상 전개 방식 중 '기승전결'에 관한 문제",
    11: "시상 전개 방식 중 '선경후정'에 관한 문제",
    12: "시를 비교해 '공통적인 중심 주제'를 찾는 문제",
    13: "시 속에 나타난 '시어의 의미'를 판단하는 문제",
    14: "시 속에 나타난 '화자의 감정 이입 대상'을 찾는 문제",

    15: "어문 규정 '제20항'의 접미사가 붙은 단어의 표기법 문제",
    16: "어문 규정 '제21항'의 접미사가 붙은 단어의 표기법 문제",
    17: "어문 규정 '제19항 의 접미사가 붙은 단어의 표기법 문제",
    18: "어문 규정 '제25항'의 '-하다'가 붙는 어근에서 파생된 부사의 표기법 문제"
}

In [3]:
def is_empty(value):
    return pd.isna(value) or str(value).strip() == ""


def to_int(value):
    if is_empty(value):
        return 0
    return int(float(value))


def parse_numbers(value):
    if is_empty(value):
        return []

    text = str(value).replace(" ", "")
    return [int(float(x)) for x in text.split(",") if x != ""]


def create_wrong_text(row):
    texts = []

    # 1~3번 독서
    r13 = to_int(row["독서1~3"])
    if r13 > 0:
        texts.append(f"이번 주에 읽어야 하는 책의 내용을 활용한 {r13}문제를 틀렸습니다.")

    # 4~6 독서 / 7~9 비문학 통합 처리
    reading = to_int(row["독서4~6"])
    nonfiction = to_int(row["비문학"])
    total = reading + nonfiction

    if total > 0:
        if reading > 0 and nonfiction > 0:
            area = "독서, 비문학"
        elif reading > 0:
            area = "독서"
        else:
            area = "비문학"

        texts.append(f"주어진 지문 안에서 근거를 찾는 문제({area}) 부분에서 {total}개를 틀렸습니다.")

    # 문학
    lit_nums = parse_numbers(row["문학"])
    if lit_nums:
        lit_parts = [f"{question_desc[n]} 1개" for n in lit_nums if n in question_desc]
        if lit_parts:
            texts.append("문학은 " + ", ".join(lit_parts) + "를 틀렸습니다.")

    # 문법
    gram_nums = parse_numbers(row["문법"])
    if gram_nums:
        gram_parts = [f"{question_desc[n]} 1개" for n in gram_nums if n in question_desc]
        if gram_parts:
            texts.append("문법은 " + ", ".join(gram_parts) + "를 틀렸습니다.")

    # 어휘
    vocab_parts = []

    if not is_empty(row["19번"]):
        vocab_parts.append("사자성어의 뜻을 찾는 1문제")

    wrong20 = to_int(row["20번"])
    if wrong20 > 0:
        vocab_parts.append(f"단어의 뜻을 찾는 {wrong20}문제")

    if vocab_parts:
        texts.append("어휘는 " + ", ".join(vocab_parts) + "를 틀렸습니다.")

    return " ".join(texts)

In [4]:
result_texts = []

for _, row in df.iterrows():
    base_text = str(row["최종문자"]).strip()
    wrong_text = create_wrong_text(row)

    if wrong_text:
        marker = "\n▷ 과 제"

        if marker in base_text:
            base_text = base_text.replace(
                marker,
                "\n→ " + wrong_text + "\n\n▷ 과 제",
                1
            )

    result_texts.append(base_text)

In [5]:
from datetime import datetime
from pathlib import Path

# 저장 폴더
save_dir = Path(r"C:\Users\9191h\OneDrive\0. Work\2. 국어 강사\0. 학생 관리\[중1] - 1학기 진도 사항\weekly_result")

# 폴더 없으면 자동 생성

save_dir.mkdir(parents=True, exist_ok=True)

# 현재 시간
now = datetime.now()

weekday_kr = ["월", "화", "수", "목", "금", "토", "일"][now.weekday()]

file_name = now.strftime(f"%Y-%m-%d_{weekday_kr}_%H시%M분") + ".txt"

# 최종 저장 경로
save_path = save_dir / file_name

# 학생별 문자 합치기
class_text = "\n\n------------------------------\n\n".join(result_texts)

# 저장
with open(save_path, "w", encoding="utf-8") as f:
    f.write(class_text)

print(f"완료: {save_path}")

완료: C:\Users\9191h\OneDrive\0. Work\2. 국어 강사\0. 학생 관리\[중1] - 1학기 진도 사항\weekly_result\2026-06-21_일_12시34분.txt
